# LAB 03
## JESUS ALEJANDRO TORAL IÑIGUEZ

In [6]:
import sys
import os

notebook_path = os.path.abspath('')
module_path = os.path.abspath(os.path.join(notebook_path, '../../src'))

from AlexTModule.spark_utils import SparkUtils


spark_url = "local[*]"
app_name = "Lab 03"
su = SparkUtils(app_name, spark_url)
su

In [2]:
import os
os.listdir("/opt/spark/work-dir/data/qcommerce/")

['qcommerce.csv']

In [3]:

"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,
Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [("Order_ID", "long"),
                ("Company", "string"),
                ("City", "string"), 
                ("Customer_Age", "int"),
                ("Order_Value", "float"),
                ("Delivery_Time_Min", "float"),
                ("Distance_Km", "float"),
                ("Items_Count", "float"),
                ("Product_Category", "string"),
                ("Payment_Method", "string"),
                ("Customer_Rating", "float"),
                ("Discount_Applied", "float"),
                ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/qcommerce.csv")

qcommerce_df.show(5)

qcommerce_df.printSchema()

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3.2|
| 1000003|Flipkart Minute

In [4]:
from pyspark.sql.functions import when,count,isnull

In [5]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c),c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

[Stage 1:====>                                                    (1 + 11) / 12]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
qcommerce_df_nulls = SparkUtils.count_nulls(qcommerce_df)
qcommerce_df_nulls.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [7]:
qcommerce_wo_nulls = qcommerce_df.dropna()
print(f"size of data frame after removing nulls: {qcommerce_wo_nulls.count()}")

size of data frame after removing nulls: 780992


In [8]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
'City': 'Uknown',
'Items_Count': 0.0,
'Customer_Rating': 0.0,
'Delivery_Partner_Rating': 0.0,
'Customer_Age' : 0.0

})

print(f"size of data frame after removing nulls with 'fillna': {qcommerce_wo_nulls_fillna.count()}")

size of data frame after removing nulls with 'fillna': 1000000


In [9]:
from pyspark.sql. functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [10]:
from pyspark. sql. functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)



+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

## TASK 1

Create Delivery_SLA with withColumn + when:
Delivery_Time_Min ≤ 15 → "FAST", 15--20 → "ON_TIME", > 20 →
"LATE"
filter only Delivery_SLA = "LATE" and orderBy Delivery_Time_Min (desc).
Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA

In [20]:
from pyspark.sql.functions import col, when

qcommerce_df_tasks1 = qcommerce_tax_df.withColumn("Delivery_SLA", 
    when(col("Delivery_Time_Min") <= 15, "FAST") \
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME") \
    .when(col("Delivery_Time_Min") > 20, "LATE")
    ) \
    .filter(col("Delivery_SLA") == "LATE") \
    .orderBy(col("Delivery_Time_Min"), ascending=False)

qcommerce_df_tasks1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA","Customer_Age").show()


+--------+--------+--------+-----------------+------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|Customer_Age|
+--------+--------+--------+-----------------+------------+------------+
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|          54|
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|          41|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|          32|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|          20|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|          42|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|          20|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|          54|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|          40|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|          43|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|          39|
| 1059671|Jio Mart|Haridwar|           39.982|     

## Task 2

Create Effective_Order_Value = Order_Value * (1 - Discount_Applied).
Create Price_Bucket with when:
< 200 → "LOW", 200--500 → "MEDIUM", > 500 → "HIGH"
groupBy(Price_Bucket) and compute count(*) and avg(Effective_Order_Value).
orderBy average effective value (desc).

In [21]:
from pyspark.sql.functions import avg, count
qcommerce_df_tasks2 = qcommerce_df_tasks1.withColumn("Effective_Order_Value", col("Order_Value") * (1-col("Discount_Applied"))) \
    .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
        .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
        .when(col("Effective_Order_Value") > 500, "HIGH")) \
    .groupBy("Price_Bucket").agg(
        count("*").alias("Count_Price_Bucket"),
        avg("Effective_Order_Value").alias("Avg_Effective_Value")
    ).orderBy("Avg_Effective_Value", ascending=False)
qcommerce_df_tasks2.show()

+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Value|
+------------+------------------+-------------------+
|        HIGH|             56013|   707.219837039914|
|      MEDIUM|             59824| 353.49134017350553|
|         LOW|            143811|  25.36497016341368|
+------------+------------------+-------------------+



## Task 3



    Create Age_Group using withColumn + when:

        < 25 → “YOUNG”

        25–44 → “ADULT”

        ≥ 45 → “SENIOR”

    Filter invalid ages (nulls, < 0, or > 100).

    Group by Age_Group and Product_Category and compute:

        orders = count(*)

        avg_order_value = avg(Order_Value)

    Order by Age_Group (ascending) and orders (descending) to find top categories per segment.


In [23]:
qcommerce_df_taks3 = (
    qcommerce_df_tasks1
    .filter(
        (col("Customer_Age").isNotNull()) & 
        (col("Customer_Age") >= 0) & 
        (col("Customer_Age") <= 100)
    )
    .withColumn("Age_Group", 
        when(col("Customer_Age") <= 25, "YOUNG")
        .when((col("Customer_Age") > 25) & (col("Customer_Age") <= 44), "ADULT")
        .otherwise("SENIOR")
    )
    .groupBy("Age_Group", "Product_Category")
    .agg(
        count("*").alias("orders"),
        avg("Order_Value").alias("avg_order_value")
    )
    .orderBy(col("Age_Group").asc(), col("orders").desc())
)

qcommerce_df_taks3.show()

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|orders|   avg_order_value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17078|495.25267501591605|
|    ADULT|              Dairy| 16901|  502.789107149631|
|    ADULT|          Household| 16863| 497.2638757191384|
|    ADULT|Fruits & Vegetables| 16823| 494.2270302383921|
|    ADULT|             Snacks| 16809| 499.4955506958792|
|    ADULT|          Beverages| 16753| 496.3202605360495|
|    ADULT|          Groceries| 16657| 491.6545542916051|
|   SENIOR|          Groceries| 13440|500.87560482649576|
|   SENIOR|      Personal Care| 13253| 501.4177858687559|
|   SENIOR|             Snacks| 13250|502.81940932795686|
|   SENIOR|              Dairy| 13176|498.58672649583895|
|   SENIOR|          Household| 13163|  489.620170382284|
|   SENIOR|          Beverages| 13118|492.35374550340305|
|   SENIOR|Fruits & Vegetables| 12955|498.62873171412866|
|    YOUNG|   